In [ ]:
"""Startet eine einzelne Simulation mit PyBullet-GUI zum Debuggen.

Aufruf (aus dem backend-Ordner):
    .venv\\Scripts\\python.exe debug_sim_gui.py [stl_pfad] [anzahl]

Ohne Argumente: neueste STL aus storage/converted, 40 Artikel.
Man sieht die Collision-Geometrie (VHACD-Teile), nicht das Render-Mesh --
genau das, womit die Physik tatsaechlich rechnet.
"""

import sys
from pathlib import Path

import trimesh

from app.core.config import CONVERTED_DIR
from app.geometry import compute_scaled_volume_mm3, ensure_vhacd_collision_mesh
from app.packing.models import Box, Item
from app.simulation.sim import PackagingSimulation, SimulationConfig


def newest_stl() -> Path:
    stls = sorted(
        Path(CONVERTED_DIR).glob("*.stl"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if not stls:
        raise SystemExit("Keine STL in storage/converted gefunden.")
    return stls[0]


def main() -> None:
    stl = Path(sys.argv[1]) if len(sys.argv) > 1 else newest_stl()
    quantity = int(sys.argv[2]) if len(sys.argv) > 2 else 40

    # Artikelmasse aus der STL ableiten (mm)
    mesh = trimesh.load(str(stl), force="mesh")
    extents = mesh.bounding_box.extents
    item = Item(
        length=float(extents[0]),
        width=float(extents[1]),
        height=float(extents[2]),
        weight=6.0,  # Gramm, Debug-Default (entspricht item_mass 0.006 kg)
    )

    config = SimulationConfig(
        item=item,
        item_quantity=quantity,
        boxes=[Box(name="Debug-Box", length=200, width=200, height=200, capacityLHM=0)],
        mesh_volume=compute_scaled_volume_mm3(stl),
        stl_file=str(stl),
        collision_file=str(ensure_vhacd_collision_mesh(stl)),
        use_gui=True,
        parallel_simulations=False,
    )

    print(f"STL: {stl.name}")
    print(f"Artikel: {item.length:.1f} x {item.width:.1f} x {item.height:.1f} mm, {quantity} Stueck")
    print("Box: 200 x 200 x 200 mm\n")

    PackagingSimulation(config).run()


if __name__ == "__main__":
    main()
